# Perbandingan model: hasil validation terbaru

Jalankan **lima cell Code pertama** secara berurutan. Hasilnya mencakup
14 run training dan satu evaluasi tambahan pada threshold pilihan 0,7.
Cell Code ketiga menampilkan tabel terbaru dan cell Code kelima
mendokumentasikannya ke W&B.

- Model yang sudah tersimpan digunakan kembali; tidak ada training ulang.
- Pembagian training, validation, dan test mengikuti manifest proyek.
- Semua nilai dalam tabel merupakan hasil **validation**, bukan test atau CV.
- Threshold dicantumkan per baris karena tidak semua eksperimen memakainya sama.
- Bagian pencarian threshold awal tetap tersedia di bawah sebagai riwayat.

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
import wandb
import yaml

# Menemukan folder proyek dari lokasi notebook dijalankan.
project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "src" / "data_loader.py").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Buka notebook dari folder ecommerce-purchase-prediction "
        "atau dari folder notebooks di dalamnya."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_raw_data, validate_manifest, get_split
from src.evaluation import evaluate_model

PROJECT_PATH = "elsaelisa/ecommerce-purchase-prediction"
api = wandb.Api()

print("Folder proyek:", project_root)

Folder proyek: c:\Users\elsae\Downloads\ecommerce-purchase-prediction


In [2]:
# Run training awal dan run pengembangan berikutnya.
run_ids = [
    "g79o1osn",  # Logistic Regression — semua fitur
    "sviknlww",  # Logistic Regression — tanpa PageValues
    "ve2ag50y",  # Random Forest — semua fitur
    "ndx4hc7p",  # Random Forest — tanpa PageValues
    "u1b7ipvz",  # XGBoost awal — semua fitur
    "3h89h2ft",  # XGBoost awal — tanpa PageValues
    "5at54aps",  # DummyClassifier
    "vqgxi5ab",  # Random Forest — class weight balanced
    "wrtlzf8i",  # Random Forest — SMOTENC
    "22cse45y",  # XGBoost tuned — semua fitur
    "zruv9kiq",  # XGBoost tuned — tanpa PageValues
    "wxyu3tcu",  # CatBoost — semua fitur
    "pz7e2gh4",  # CatBoost — tanpa PageValues
    "d9n08p0g",  # XGBoost — scale_pos_weight otomatis, threshold awal 0,3
]

metric_names = [
    "precision",
    "recall",
    "f1",
    "average_precision",
    "roc_auc",
]

display_columns = [
    "experiment", "n_features", "threshold", *metric_names
]

records = []
runs_by_id = {}

for run_id in run_ids:
    source_run = api.run(f"{PROJECT_PATH}/{run_id}")
    runs_by_id[run_id] = source_run
    config = source_run.config

    row = {
        "run_id": source_run.id,
        "experiment": source_run.name,
        "model": config["model"]["name"],
        "n_features": len(config["feature_columns"]),
        "threshold": float(config["evaluation"]["threshold"]),
        "run_url": source_run.url,
        "evaluation_split": "validation",
        "metrics_source": "wandb_training_run_summary",
        "threshold_source": "training_run_config",
    }

    for metric in metric_names:
        key = f"validation/{metric}"
        value = source_run.summary.get(key)
        if value is None:
            raise ValueError(f"Metrik {key} tidak ditemukan pada run {run_id}.")
        row[metric] = float(value)

    records.append(row)

results_df = pd.DataFrame(records)

print("Jumlah hasil dari run training:", len(results_df))
print("Baris threshold pilihan 0,7 ditambahkan pada cell Code berikutnya.")
display(results_df[display_columns].round(4))

Jumlah hasil dari run training: 14
Baris threshold pilihan 0,7 ditambahkan pada cell Code berikutnya.


,experiment,n_features,threshold,precision,recall,f1,average_precision,roc_auc
0,lr_all_features,17,0.5,0.7179,0.3784,0.4956,0.6232,0.8907
1,lr_without_pagevalues,16,0.5,0.4483,0.0351,0.0652,0.2935,0.7360
2,rf_all_features,17,0.5,0.7117,0.5405,0.6144,0.7243,0.9236
3,rf_without_pagevalues,16,0.5,0.4286,0.0892,0.1477,0.3276,0.7593
4,xgb_all_features,17,0.5,0.6759,0.5919,0.6311,0.7285,0.9308
5,xgb_without_pagevalues,16,0.5,0.4184,0.1108,0.1752,0.3243,0.7713
6,dummy_prior,17,0.5,0.0000,0.0000,0.0000,0.1508,0.5000
7,rf_all_features_balanced,17,0.5,0.7331,0.5270,0.6132,0.7140,0.9214
8,rf_all_features_smotenc,17,0.5,0.5833,0.7189,0.6441,0.6326,0.9137
9,xgb_tuned_all_features,17,0.3,0.5726,0.7676,0.6559,0.7419,0.9331


## Menambahkan hasil threshold pilihan

Run `d9n08p0g` mencatat threshold awal **0,3**. Cell berikut membaca
`configs/weighted_candidate_selection.yaml`, memuat pipeline tersimpan,
dan menghitung ulang metrik **validation** pada threshold yang sudah dipilih.
Mengganti angka threshold di tabel saja tidak menghitung ulang metriknya.

Untuk keputusan terakhir kita, file pilihan tersebut berisi threshold **0,7**.
Model serta threshold yang digunakan akan dicetak saat cell dijalankan.

In [3]:
# Membaca keputusan threshold yang sudah dibuat pada tahap pengembangan.
candidate_selection_path = (
    project_root / "configs" / "weighted_candidate_selection.yaml"
)

if not candidate_selection_path.is_file():
    raise FileNotFoundError(
        f"File pilihan kandidat belum ditemukan: {candidate_selection_path}"
    )

with candidate_selection_path.open("r", encoding="utf-8") as file:
    candidate_selection = yaml.safe_load(file)

candidate_run_path = candidate_selection["source_run_path"]
candidate_project, candidate_run_id = candidate_run_path.rsplit("/", 1)

if candidate_project != PROJECT_PATH or candidate_run_id not in runs_by_id:
    raise ValueError("Run kandidat harus berasal dari daftar run proyek di atas.")

candidate_threshold = float(candidate_selection["threshold"])
if not 0 < candidate_threshold < 1:
    raise ValueError("Threshold kandidat harus berada di antara 0 dan 1.")

candidate_run = runs_by_id[candidate_run_id]
candidate_config = candidate_run.config

candidate_df, candidate_manifest = load_raw_data(candidate_config["data"])
validate_manifest(candidate_df, candidate_manifest)

candidate_X_val, candidate_y_val = get_split(
    candidate_df,
    candidate_manifest,
    split_name="validation",
    target_column=candidate_config["data"]["target"],
    exclude_columns=candidate_config["features"]["exclude_columns"],
)

candidate_pipeline_path = (
    project_root
    / candidate_config["output_dir"]
    / candidate_run_id
    / "pipeline.joblib"
)

if not candidate_pipeline_path.is_file():
    raise FileNotFoundError(f"Pipeline belum ditemukan: {candidate_pipeline_path}")

candidate_pipeline = joblib.load(candidate_pipeline_path)
candidate_metrics, _, _ = evaluate_model(
    candidate_pipeline,
    candidate_X_val,
    candidate_y_val,
    threshold=candidate_threshold,
)

candidate_row = {
    "run_id": candidate_run.id,
    "experiment": (
        f"{candidate_run.name}_threshold_{candidate_threshold:g}"
    ),
    "model": candidate_config["model"]["name"],
    "n_features": candidate_X_val.shape[1],
    "threshold": candidate_threshold,
    "run_url": candidate_run.url,
    "evaluation_split": "validation",
    "metrics_source": "saved_pipeline_validation_evaluation",
    "threshold_source": "configs/weighted_candidate_selection.yaml",
    **candidate_metrics,
}

# Menjaga agar cell ini bisa diulang tanpa menggandakan baris yang sama.
same_candidate = (
    results_df["run_id"].eq(candidate_run.id)
    & results_df["threshold"].eq(candidate_threshold)
)
results_df = pd.concat(
    [results_df.loc[~same_candidate], pd.DataFrame([candidate_row])],
    ignore_index=True,
)

print("Model kandidat:", candidate_run.name)
print("Threshold kandidat:", candidate_threshold)
print("Ukuran validation:", candidate_X_val.shape)
print("Jumlah baris perbandingan:", len(results_df))

display(
    results_df[display_columns]
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
    .round(4)
)

Model kandidat: xgb_all_features_auto_weight
Threshold kandidat: 0.7
Ukuran validation: (2454, 17)
Jumlah baris perbandingan: 15


,experiment,n_features,threshold,precision,recall,f1,average_precision,roc_auc
0,xgb_all_features_auto_weight_threshold_0.7,17,0.7,0.6040,0.7378,0.6642,0.7390,0.9324
1,xgb_tuned_all_features,17,0.3,0.5726,0.7676,0.6559,0.7419,0.9331
2,catboost_all_features,17,0.3,0.5768,0.7405,0.6485,0.7401,0.9299
3,rf_all_features_smotenc,17,0.5,0.5833,0.7189,0.6441,0.6326,0.9137
4,xgb_all_features,17,0.5,0.6759,0.5919,0.6311,0.7285,0.9308
5,rf_all_features,17,0.5,0.7117,0.5405,0.6144,0.7243,0.9236
6,rf_all_features_balanced,17,0.5,0.7331,0.5270,0.6132,0.7140,0.9214
7,xgb_all_features_auto_weight,17,0.3,0.4057,0.9297,0.5649,0.7390,0.9324
8,lr_all_features,17,0.5,0.7179,0.3784,0.4956,0.6232,0.8907
9,catboost_without_pagevalues,16,0.3,0.3646,0.3784,0.3714,0.3241,0.7690


In [4]:
# Menyimpan rekap terbaru setelah hasil threshold pilihan ditambahkan.
report_dir = project_root / "outputs" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

csv_path = report_dir / "validation_model_comparison.csv"
results_df.to_csv(csv_path, index=False)

print("Jumlah hasil:", len(results_df))
print("Tabel disimpan:", csv_path)

Jumlah hasil: 15
Tabel disimpan: c:\Users\elsae\Downloads\ecommerce-purchase-prediction\outputs\reports\validation_model_comparison.csv


In [5]:
# Mendokumentasikan seluruh tabel sebagai run analisis baru.
# Threshold setiap eksperimen mengikuti kolom threshold pada tabel.
with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name="model_comparison_updated",
    job_type="analysis",
    mode="online",
    config={
        "source_run_ids": run_ids,
        "evaluation_split": "validation",
        "threshold_policy": "per_row",
        "thresholds_in_report": sorted(
            results_df["threshold"].unique().tolist()
        ),
        "candidate_selection": candidate_selection,
    },
) as comparison_run:
    comparison_run.log({
        "comparison/validation_results": wandb.Table(
            dataframe=results_df
        )
    })

    best_row = results_df.loc[results_df["f1"].idxmax()]
    comparison_run.summary["best_validation_f1"] = float(best_row["f1"])
    comparison_run.summary["best_validation_experiment"] = best_row["experiment"]
    comparison_run.summary["best_validation_threshold"] = float(
        best_row["threshold"]
    )

    artifact = wandb.Artifact(
        name="validation-model-comparison",
        type="evaluation",
    )
    artifact.add_file(str(csv_path))
    artifact.add_file(str(candidate_selection_path))
    comparison_run.log_artifact(artifact)

    print("Link dokumentasi:", comparison_run.url)

wandb: Currently logged in as: rifayel-126140185 (elsaelisa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Link dokumentasi: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/wgvare3x


best_validation_experiment,xgb_all_features_aut...
best_validation_f1,0.66423
best_validation_threshold,0.7


## Riwayat pencarian threshold awal

Cell di bawah adalah percobaan awal XGBoost dan Random Forest yang sudah
pernah dijalankan. Untuk memperbarui tabel terbaru dan dokumentasi W&B,
cukup jalankan lima cell Code di atas. Bagian riwayat tidak perlu dijalankan ulang.

In [6]:
import sys
import joblib

# Memungkinkan notebook mengimpor kode dari folder src.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import (
    load_raw_data,
    validate_manifest,
    get_split,
)
from src.evaluation import evaluate_model

# Membaca konfigurasi asli XGBoost semua fitur.
selected_run_id = "u1b7ipvz"
selected_run = api.run(f"{PROJECT_PATH}/{selected_run_id}")
selected_config = selected_run.config

df, manifest = load_raw_data(selected_config["data"])
validate_manifest(df, manifest)

X_val, y_val = get_split(
    df,
    manifest,
    split_name="validation",
    target_column=selected_config["data"]["target"],
    exclude_columns=selected_config["features"]["exclude_columns"],
)

# Memuat pipeline yang sudah dilatih.
pipeline_path = (
    project_root
    / selected_config["output_dir"]
    / selected_run_id
    / "pipeline.joblib"
)

selected_pipeline = joblib.load(pipeline_path)

print("Model:", selected_run.name)
print("Ukuran validation:", X_val.shape)

Model: xgb_all_features
Ukuran validation: (2454, 17)


In [7]:
threshold_records = []

for threshold in [0.5, 0.4, 0.3, 0.2, 0.1]:
    metrics, predictions, _ = evaluate_model(
        selected_pipeline,
        X_val,
        y_val,
        threshold=threshold,
    )

    threshold_records.append({
        "threshold": threshold,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "predicted_purchases": int(predictions.sum()),
    })

threshold_results_df = pd.DataFrame(threshold_records)

display(threshold_results_df.round(4))

,threshold,precision,recall,f1,predicted_purchases
0,0.5,0.6759,0.5919,0.6311,324
1,0.4,0.6150,0.6649,0.6390,400
2,0.3,0.5771,0.7486,0.6518,480
3,0.2,0.5396,0.8108,0.6479,556
4,0.1,0.4568,0.9000,0.6060,729


In [8]:
# Mengambil threshold dengan F1 tertinggi dari percobaan.
best_index = threshold_results_df["f1"].idxmax()
best_result = threshold_results_df.loc[best_index]

with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name="xgb_all_features_threshold_search",
    job_type="threshold_tuning",
    mode="online",
    config={
        "source_run_id": selected_run_id,
        "evaluation_split": "validation",
        "selection_metric": "f1",
        "tested_thresholds": (
            threshold_results_df["threshold"].tolist()
        ),
    },
) as run:

    run.log({
        "threshold_comparison": wandb.Table(
            dataframe=threshold_results_df
        )
    })

    run.summary["best_tested_threshold"] = float(
        best_result["threshold"]
    )

    for metric in ["precision", "recall", "f1"]:
        run.summary[f"validation/{metric}"] = float(
            best_result[metric]
        )

    print("Kandidat threshold:", best_result["threshold"])
    print("Link W&B:", run.url)

Kandidat threshold: 0.3
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/5clua3pe


best_tested_threshold,0.3
validation/f1,0.65176
validation/precision,0.57708
validation/recall,0.74865


# menguji threshold Random Forest awal

In [9]:
from pathlib import Path
import sys

import joblib
import pandas as pd
import wandb

project_root = Path(
    r"C:\Users\elsae\Downloads\ecommerce-purchase-prediction"
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import (
    load_raw_data,
    validate_manifest,
    get_split,
)
from src.evaluation import evaluate_model

# Mengambil konfigurasi Random Forest awal.
rf_run_id = "ve2ag50y"
rf_run = wandb.Api().run(
    f"elsaelisa/ecommerce-purchase-prediction/{rf_run_id}"
)
rf_config = rf_run.config

rf_df, rf_manifest = load_raw_data(rf_config["data"])
validate_manifest(rf_df, rf_manifest)

rf_X_val, rf_y_val = get_split(
    rf_df,
    rf_manifest,
    split_name="validation",
    target_column=rf_config["data"]["target"],
    exclude_columns=rf_config["features"]["exclude_columns"],
)

# Memuat pipeline tersimpan, tanpa training ulang.
rf_pipeline = joblib.load(
    project_root
    / rf_config["output_dir"]
    / rf_run_id
    / "pipeline.joblib"
)

print("Model:", rf_run.name)
print("Validation:", rf_X_val.shape)

Model: rf_all_features
Validation: (2454, 17)


In [10]:
rf_threshold_records = []

for threshold in [0.5, 0.4, 0.3, 0.2, 0.1]:
    metrics, predictions, _ = evaluate_model(
        rf_pipeline,
        rf_X_val,
        rf_y_val,
        threshold=threshold,
    )

    rf_threshold_records.append({
        "threshold": threshold,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "predicted_purchases": int(predictions.sum()),
    })

rf_threshold_results_df = pd.DataFrame(
    rf_threshold_records
)

display(rf_threshold_results_df.round(4))

,threshold,precision,recall,f1,predicted_purchases
0,0.5,0.7117,0.5405,0.6144,281
1,0.4,0.6354,0.6595,0.6472,384
2,0.3,0.5756,0.7405,0.6478,476
3,0.2,0.5008,0.8324,0.6254,615
4,0.1,0.3588,0.9378,0.5191,967


In [11]:
best_rf_index = rf_threshold_results_df["f1"].idxmax()
best_rf_result = rf_threshold_results_df.loc[best_rf_index]

with wandb.init(
    entity="elsaelisa",
    project="ecommerce-purchase-prediction",
    name="rf_all_features_threshold_search",
    job_type="threshold_tuning",
    mode="online",
    config={
        "source_run_id": rf_run_id,
        "evaluation_split": "validation",
        "selection_metric": "f1",
        "tested_thresholds": (
            rf_threshold_results_df["threshold"].tolist()
        ),
    },
) as run:

    run.log({
        "threshold_comparison": wandb.Table(
            dataframe=rf_threshold_results_df
        )
    })

    run.summary["best_tested_threshold"] = float(
        best_rf_result["threshold"]
    )

    for metric in ["precision", "recall", "f1"]:
        run.summary[f"validation/{metric}"] = float(
            best_rf_result[metric]
        )

    print("Kandidat threshold RF:", best_rf_result["threshold"])
    print("Link W&B:", run.url)

Kandidat threshold RF: 0.3
Link W&B: https://wandb.ai/elsaelisa/ecommerce-purchase-prediction/runs/9et28u6c


best_tested_threshold,0.3
validation/f1,0.64775
validation/precision,0.57563
validation/recall,0.74054
